In [ ]:
from datetime import datetime

job_id = session.sql("SELECT UUID_STRING()").collect()[0][0]
job_name = "DWH_DIM_CUSTODIES_INFO_LOAD"
layer_name = "DWH"
start_time = datetime.now()

rows_loaded = 0
rows_failed = 0
run_status = "SUCCESS"
error_message = None

try:

    session.sql(f"""
    CREATE OR REPLACE TEMP TABLE TEMP_CUSTODIES AS
    SELECT DISTINCT
        av1.VALUE_SHORT_DESC AS LEGAL_STATUS_CODE_DESC,
        c.LEGAL_STATUS_CODE AS LEGAL_STATUS_CODE_AV_ID,
        av2.VALUE_SHORT_DESC AS END_REASON_CODE_DESC,
        c.END_REASON_CODE AS END_REASON_CODE_AV_ID,
        av3.VALUE_SHORT_DESC AS CUSTODY_TYPE_DESC,
        c.CUSTODY_TYPE AS CUSTODY_TYPE_AV_ID,
        c.DELETED_DATE,
        SHA2(CONCAT_WS('|',
            COALESCE(av1.VALUE_SHORT_DESC, ''),
            COALESCE(TO_VARCHAR(c.LEGAL_STATUS_CODE), ''),
            COALESCE(av2.VALUE_SHORT_DESC, ''),
            COALESCE(TO_VARCHAR(c.END_REASON_CODE), ''),
            COALESCE(av3.VALUE_SHORT_DESC, ''),
            COALESCE(TO_VARCHAR(c.CUSTODY_TYPE), ''),
            COALESCE(TO_VARCHAR(c.DELETED_DATE), '')
        )) AS CHECKSUM
    FROM {STG}.{STG_CUSTODIES} c
    LEFT JOIN {STG}.{STG_ALLOWABLE_VALUES} av1
        ON c.LEGAL_STATUS_CODE = av1.AV_ID
    LEFT JOIN {STG}.{STG_ALLOWABLE_VALUES} av2
        ON c.END_REASON_CODE = av2.AV_ID
    LEFT JOIN {STG}.{STG_ALLOWABLE_VALUES} av3
        ON c.CUSTODY_TYPE = av3.AV_ID
    """).collect()

    session.sql(f"""
    UPDATE {DWH}.{DIM_CUSTODIES_INFO} tgt
    SET UPDATED_DATE = CURRENT_TIMESTAMP()
    FROM TEMP_CUSTODIES src
    WHERE tgt.LEGAL_STATUS_CODE_AV_ID IS NOT DISTINCT FROM src.LEGAL_STATUS_CODE_AV_ID
      AND tgt.END_REASON_CODE_AV_ID IS NOT DISTINCT FROM src.END_REASON_CODE_AV_ID
      AND tgt.CUSTODY_TYPE_AV_ID IS NOT DISTINCT FROM src.CUSTODY_TYPE_AV_ID
      AND tgt.UPDATED_DATE IS NULL
      AND tgt.CHECKSUM <> src.CHECKSUM
    """).collect()

    insert_result = session.sql(f"""
    INSERT INTO {DWH}.{DIM_CUSTODIES_INFO}
    (
        LEGAL_STATUS_CODE_DESC,
        LEGAL_STATUS_CODE_AV_ID,
        END_REASON_CODE_DESC,
        END_REASON_CODE_AV_ID,
        CUSTODY_TYPE_DESC,
        CUSTODY_TYPE_AV_ID,
        CREATED_DATE,
        UPDATED_DATE,
        DELETED_DATE,
        CHECKSUM
    )
    SELECT
        src.LEGAL_STATUS_CODE_DESC,
        src.LEGAL_STATUS_CODE_AV_ID,
        src.END_REASON_CODE_DESC,
        src.END_REASON_CODE_AV_ID,
        src.CUSTODY_TYPE_DESC,
        src.CUSTODY_TYPE_AV_ID,
        CURRENT_TIMESTAMP(),
        NULL,
        src.DELETED_DATE,
        src.CHECKSUM
    FROM TEMP_CUSTODIES src
    WHERE NOT EXISTS
    (
        SELECT 1
        FROM {DWH}.{DIM_CUSTODIES_INFO} tgt
        WHERE tgt.CHECKSUM = src.CHECKSUM
          AND tgt.UPDATED_DATE IS NULL
    )
    """).collect()

    rows_loaded = insert_result[0][0]

    session.call(
       f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
        job_id,
        job_name,
        layer_name,
        start_time,
        datetime.now(),
        rows_loaded,
        rows_failed,
        run_status,
        None
    )

    session.call(
       f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
        run_status,
        job_name,
        layer_name,
        f"DIM_CUSTODIES_INFO load completed. Rows Loaded: {rows_loaded}"
    )

    print(f"DWH LOAD SUCCESS | Rows Loaded: {rows_loaded}")

except Exception as error:

    run_status = "FAILED"
    rows_failed = 1
    error_message = str(error)

    session.call(
       f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
        job_id,
        job_name,
        layer_name,
        start_time,
        datetime.now(),
        0,
        rows_failed,
        run_status,
        error_message
    )

    session.call(
       f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
        run_status,
        job_name,
        layer_name,
        f"DIM_CUSTODIES_INFO load failed. Error: {error_message}"
    )

    print(f"DWH LOAD FAILED: {error_message}")